# Antivirus Rating System using TOPSIS MCDM

Sistem rating antivirus berbasis metode **TOPSIS** (Technique for Order of Preference by Similarity to Ideal Solution) menggunakan data dari **AV-TEST Institute**.

## Metodologi
1. **Data Source**: AV-TEST.org (Protection, Performance, Usability scores)
2. **Normalization**: Vector Normalization
3. **MCDM Method**: TOPSIS
4. **Kriteria**: Protection (benefit), Performance (benefit), Usability (benefit)

---

## 1. Setup Environment

In [ ]:
!pip install pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded.')

## 2. Setup Kaggle API & Pull Dataset

Upload `kaggle.json` saat cell ini dijalankan.

In [ ]:
!pip install kaggle -q
from google.colab import files

print('Upload file kaggle.json Anda:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API configured.')

In [ ]:
# Ganti YOUR_USERNAME/antivirus-test-ratings dengan dataset slug Anda
# Format: username/dataset-slug
KAGGLE_DATASET = 'razialfarizi/antivirus-test-ratings'  # Ganti dengan dataset Anda

!mkdir -p data/raw
!kaggle datasets download -d {KAGGLE_DATASET} -p ./data/raw --unzip

import os
csv_files = [f for f in os.listdir('data/raw') if f.endswith('.csv')]
print(f'CSV files found: {csv_files}')

## 3. Load & Explore Data

In [ ]:
# Load dataset
df = pd.read_csv('data/raw/av_test_ratings.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nPlatforms: {df["platform"].unique()}')
print(f'Products: {df["antivirus_name"].nunique()}')
print(f'Periods: {sorted(df["test_period"].unique())}')
print(f'\nFirst 10 rows:')
df.head(10)

In [ ]:
# Statistical summary
df.groupby('platform')[['protection', 'performance', 'usability']].describe().round(2)

## 4. TOPSIS Implementation

### Rumus TOPSIS:
1. **Vector Normalization**: $norm_{ij} = \frac{x_{ij}}{\sqrt{\sum x_{ij}^2}}$
2. **Weighted Matrix**: $v_{ij} = w_j \times norm_{ij}$
3. **Ideal Solution**: $A^+ = \max(v_{ij})$ untuk benefit, $A^- = \min(v_{ij})$
4. **Euclidean Distance**: $D^+ = \sqrt{\sum(v_{ij} - A^+)^2}$, $D^- = \sqrt{\sum(v_{ij} - A^-)^2}$
5. **Preference Score**: $V_i = \frac{D^-_i}{D^+_i + D^-_i}$

In [ ]:
CRITERIA = ['protection', 'performance', 'usability']

def normalize_vector(df, cols):
    """Vector normalization: norm_ij = x_ij / sqrt(sum(x_ij^2))"""
    norm = df.copy()
    for col in cols:
        denom = np.sqrt(np.sum(df[col]**2))
        norm[col] = df[col] / denom if denom > 0 else 0
    return norm

def calculate_topsis(df, weights, criteria_types):
    """Calculate TOPSIS scores and rankings."""
    # Aggregate per product
    agg = df.groupby(['antivirus_name', 'platform']).agg(
        protection=('protection', 'mean'),
        performance=('performance', 'mean'),
        usability=('usability', 'mean'),
        periods_tested=('test_period', 'count')
    ).reset_index()
    
    # Step 1: Normalize
    norm = normalize_vector(agg, CRITERIA)
    
    # Step 2: Weighted normalized matrix
    weighted = norm.copy()
    for i, col in enumerate(CRITERIA):
        weighted[col] = norm[col] * weights[i]
    
    # Step 3: Ideal solutions
    a_plus, a_minus = [], []
    for i, col in enumerate(CRITERIA):
        if criteria_types[i] == 'benefit':
            a_plus.append(weighted[col].max())
            a_minus.append(weighted[col].min())
        else:
            a_plus.append(weighted[col].min())
            a_minus.append(weighted[col].max())
    a_plus, a_minus = np.array(a_plus), np.array(a_minus)
    
    # Step 4: Euclidean distances
    d_plus = [np.sqrt(np.sum((row[CRITERIA].values.astype(float) - a_plus)**2)) for _, row in weighted.iterrows()]
    d_minus = [np.sqrt(np.sum((row[CRITERIA].values.astype(float) - a_minus)**2)) for _, row in weighted.iterrows()]
    
    # Step 5: Preference scores
    scores = [dm/(dp+dm) if (dp+dm) > 0 else 0.5 for dp, dm in zip(d_plus, d_minus)]
    
    result = agg.copy()
    result['rating_score'] = scores
    result['rank'] = result['rating_score'].rank(ascending=False, method='min').astype(int)
    
    return result.sort_values('rank').reset_index(drop=True)

print('TOPSIS functions defined.')

## 5. Run TOPSIS Rating

In [ ]:
# Konfigurasi bobot
WEIGHTS = [0.50, 0.20, 0.30]  # Protection, Performance, Usability
CRITERIA_TYPES = ['benefit', 'benefit', 'benefit']

print(f'Weights: Protection={WEIGHTS[0]}, Performance={WEIGHTS[1]}, Usability={WEIGHTS[2]}')
print(f'Criteria types: {CRITERIA_TYPES}\n')

# Filter by platform
platform = 'Windows'  # Ganti ke 'macOS' atau 'Android' atau 'all'
if platform != 'all':
    df_platform = df[df['platform'] == platform]
else:
    df_platform = df

# Run TOPSIS
result = calculate_topsis(df_platform, WEIGHTS, CRITERIA_TYPES)

print(f'=== RATING RESULTS ({platform}) ===\n')
result[['antivirus_name', 'platform', 'protection', 'performance', 'usability', 'periods_tested', 'rating_score', 'rank']]

## 6. Visualization

In [ ]:
# Bar chart - TOPSIS Rating Ranking
fig, ax = plt.subplots(figsize=(12, 7))
sorted_df = result.sort_values('rating_score', ascending=True)
colors = plt.cm.RdYlGn(sorted_df['rating_score'].values / sorted_df['rating_score'].max())

bars = ax.barh(sorted_df['antivirus_name'], sorted_df['rating_score'], color=colors)
ax.set_xlabel('TOPSIS Preference Score (V)', fontsize=12)
ax.set_title(f'Antivirus TOPSIS Rating Ranking - {platform}', fontsize=14, fontweight='bold')
ax.set_xlim(0, 1.05)

for bar, score in zip(bars, sorted_df['rating_score']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, f'{score:.4f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('ranking_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print('Chart saved: ranking_chart.png')

In [ ]:
# Radar chart for top product
top_product = result.iloc[0]['antivirus_name']
product_row = result[result['antivirus_name'] == top_product]
avg_row = result[CRITERIA].mean()

labels = ['Protection', 'Performance', 'Usability']
product_scores = product_row[CRITERIA].values.flatten()
angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()

product_plot = np.concatenate((product_scores, [product_scores[0]]))
avg_plot = np.concatenate((avg_row.values, [avg_row.values[0]]))
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, product_plot, 'o-', linewidth=2, label=top_product)
ax.fill(angles, product_plot, alpha=0.25)
ax.plot(angles, avg_plot, 'o--', linewidth=1.5, label='Average', color='gray')
ax.fill(angles, avg_plot, alpha=0.1, color='gray')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 6.5)
ax.set_title(f'Profile: {top_product}', fontsize=13, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig('radar_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print('Chart saved: radar_chart.png')

## 7. Sensitivity Analysis

Menguji stabilitas ranking dengan variasi bobot berbeda.

In [ ]:
# Definisi skenario
scenarios = {
    'Default (50/20/30)': ([0.50, 0.20, 0.30], ['benefit', 'benefit', 'benefit']),
    'Protection-heavy (70/15/15)': ([0.70, 0.15, 0.15], ['benefit', 'benefit', 'benefit']),
    'Balanced (33/33/34)': ([0.33, 0.33, 0.34], ['benefit', 'benefit', 'benefit']),
    'Performance-focus (30/50/20)': ([0.30, 0.50, 0.20], ['benefit', 'benefit', 'benefit']),
}

# Run sensitivity analysis
sens_results = {}
for name, (weights, crit_types) in scenarios.items():
    sens_results[name] = calculate_topsis(df_platform, weights, crit_types)

# Display comparison
for name, res in sens_results.items():
    print(f'\n=== {name} ===')
    print(res[['antivirus_name', 'rating_score', 'rank']].head(10).to_string(index=False))

In [ ]:
# Sensitivity analysis bar chart
scenario_names = list(scenarios.keys())
all_products = set()
for res in sens_results.values():
    all_products.update(res['antivirus_name'].tolist())

rank_data = {}
for scenario, res in sens_results.items():
    ranks = dict(zip(res['antivirus_name'], res['rank']))
    rank_data[scenario] = ranks

avg_ranks = {}
for prod in all_products:
    ranks = [rank_data[s].get(prod, len(all_products)+1) for s in scenario_names]
    avg_ranks[prod] = np.mean(ranks)

top_products = sorted(avg_ranks.keys(), key=lambda x: avg_ranks[x])[:5]

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(scenario_names))
width = 0.15

for i, prod in enumerate(top_products):
    ranks = [rank_data[s].get(prod, len(all_products)+1) for s in scenario_names]
    ax.bar(x + i*width, ranks, width, label=prod)

ax.set_xlabel('Weight Scenario', fontsize=12)
ax.set_ylabel('Rank', fontsize=12)
ax.set_title(f'Sensitivity Analysis - {platform}', fontsize=14, fontweight='bold')
ax.set_xticks(x + width*(len(top_products)-1)/2)
ax.set_xticklabels([s.split('(')[0].strip() for s in scenario_names], rotation=15, ha='right', fontsize=9)
ax.invert_yaxis()
ax.set_yticks(range(1, len(top_products)+2))
ax.legend(title='Product', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('sensitivity_chart.png', dpi=300, bbox_inches='tight')
plt.show()
print('Chart saved: sensitivity_chart.png')

## 8. Export Results

In [ ]:
# Save final results
result.to_csv('final_antivirus_rating.csv', index=False)
print('Saved: final_antivirus_rating.csv')

# Save sensitivity results
sens_dfs = []
for name, res in sens_results.items():
    tmp = res.copy()
    tmp['scenario'] = name
    sens_dfs.append(tmp)
pd.concat(sens_dfs).to_csv('sensitivity_analysis.csv', index=False)
print('Saved: sensitivity_analysis.csv')

print('\nAll done! File tersedia untuk di-download.')

In [ ]:
# Download files from Colab
from google.colab import files
files.download('final_antivirus_rating.csv')
files.download('sensitivity_analysis.csv')
files.download('ranking_chart.png')
files.download('radar_chart.png')
files.download('sensitivity_chart.png')